# Exploración de la API Data360 del Banco Mundial

Este notebook fue desarrollado por el Grupo 1 para explorar de forma progresiva la API Data360 del Banco Mundial.

Los integrantes del grupo son:
- Integrante 1: Demattei, Santiago Javier
- Integrante 2: Rigotti, Juan Pablo
- Integrante 3: Goñi, Luis Guillermo
- Integrante 4: Marzullo, Federico Agustín

---
## 1. Configuración e Imports
Instalamos e importamos las librerías necesarias y definimos la URL base de la API.

In [ ]:
import requests
import pandas as pd
import json
import altair as alt
alt.renderers.enable('colab')
alt.data_transformers.disable_max_rows()
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

BASE_URL = "https://data360api.worldbank.org"

---
## 2. Funciones para hacer peticiones a la API
A continuación, definimos funciones reutilizables que manejan errores HTTP y devuelven la respuesta en formato JSON.

In [ ]:
def get_api(endpoint, params=None):
    """Realiza una petición GET a la API y retorna el JSON de respuesta."""
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()


def post_api(endpoint, payload):
    """Realiza una petición POST a la API y retorna el JSON de respuesta."""
    url = f"{BASE_URL}{endpoint}"
    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()

---
## 3. Búsqueda general: ¿Qué bases de datos existen?
Usamos el endpoint de búsqueda `/data360/searchv2` para descubrir las bases de datos disponibles.
El parámetro `type eq 'dataset'` filtra solo resultados de tipo base de datos.

In [ ]:
payload_databases = {
    "count": True,
    "filter": "type eq 'dataset'",
    "select": "series_description/idno, series_description/name, series_description/database_id",
    "top": 1000, # numero grande para traer todas las db en una consulta
    "skip": 0
}

resp_databases = post_api("/data360/searchv2", payload_databases)

print(f"Total de bases de datos encontradas: {resp_databases.get('@odata.count', 'N/A')}")

Total de bases de datos encontradas: 160


Convertimos los resultados en un DataFrame para explorarlos de una manera más cómoda.

In [ ]:
records_db = []
for item in resp_databases.get('value', []):
    sd = item.get('series_description', {})
    records_db.append({
        'database_id': sd.get('database_id') or sd.get('idno', ''),
        'nombre': sd.get('name', ''),
        'idno': sd.get('idno', '')
    })

df_databases = pd.DataFrame(records_db).drop_duplicates(subset='database_id').reset_index(drop=True)
print(f"Bases de datos únicas: {len(df_databases)}")
display(df_databases)

Bases de datos únicas: 160


,database_id,nombre,idno
0,IMF_IRFCL,International Reserves and Foreign Currency Liquidity (IRFCL),None
1,FH_NIT,Nations in Transit,None
2,IBP_OBS,Open Budget Survey,None
3,ILO_EMP,Employment by economic activity,None
4,WB_WITS,World Integrated Trade Solution (WITS),None
...,...,...,...
155,WB_GIRG,Global Indicators of Regulatory Governance,None
156,WB_GWA,Global Wind Atlas (GWA),None
157,WB_CCKP,CCKP ERA5 Dataset,None
158,WB_LPGD,Learning Poverty Global Database,None


---
## 4. Indicadores disponibles por base de datos
El endpoint `/data360/indicators` devuelve la lista de IDs de indicadores para una base de datos dada.
Exploramos primero el WDI (World Development Indicators), que es la base más conocida del Banco Mundial.

In [ ]:
DB_ID = "WB_WDI"
resp_indicators = get_api("/data360/indicators", params={"datasetId": DB_ID})

if isinstance(resp_indicators, list): # si la respuesta es una lista directa de indicadores
    indicators_raw = resp_indicators
elif isinstance(resp_indicators, dict): # si la respuesta es un dict con una clave 'value' que contiene la lista de indicadores
    indicators_raw = resp_indicators.get('value', resp_indicators)
else:
    indicators_raw = []

print(f"Total de indicadores en '{DB_ID}': {len(indicators_raw)}")
print("\nPrimeros 10 indicadores:")
print(indicators_raw[:10])

Total de indicadores en 'WB_WDI': 1515

Primeros 10 indicadores:
['WB_WDI_AG_LND_CREL_HA', 'WB_WDI_AG_LND_PRCP_MM', 'WB_WDI_AG_LND_TOTL_K2', 'WB_WDI_AG_YLD_CREL_KG', 'WB_WDI_BM_GSR_FCTY_CD', 'WB_WDI_BM_GSR_ROYL_CD', 'WB_WDI_BM_GSR_TRVL_ZS', 'WB_WDI_BM_KLT_DINV_WD_GD_ZS', 'WB_WDI_BM_TRF_PWKR_CD_DT', 'WB_WDI_BN_FIN_TOTL_CD']


Convertimos los indicadores en un DataFrame estructurado.

In [ ]:
# ajustamos la creación del df segun el caso
if indicators_raw and isinstance(indicators_raw[0], str):
    df_indicators = pd.DataFrame({'indicator_id': indicators_raw})
elif indicators_raw and isinstance(indicators_raw[0], dict):
    df_indicators = pd.json_normalize(indicators_raw)
else:
    df_indicators = pd.DataFrame({'indicator_id': indicators_raw})

print(f"Shape del DataFrame de indicadores: {df_indicators.shape}")
print(f"Columnas: {df_indicators.columns.tolist()}")
display(df_indicators.head(20))

Shape del DataFrame de indicadores: (1515, 1)
Columnas: ['indicator_id']


,indicator_id
0,WB_WDI_AG_LND_CREL_HA
1,WB_WDI_AG_LND_PRCP_MM
2,WB_WDI_AG_LND_TOTL_K2
3,WB_WDI_AG_YLD_CREL_KG
4,WB_WDI_BM_GSR_FCTY_CD
5,WB_WDI_BM_GSR_ROYL_CD
6,WB_WDI_BM_GSR_TRVL_ZS
7,WB_WDI_BM_KLT_DINV_WD_GD_ZS
8,WB_WDI_BM_TRF_PWKR_CD_DT
9,WB_WDI_BN_FIN_TOTL_CD


---
## 5. Búsqueda de indicadores por temática
El endpoint `/data360/searchv2` permite buscar indicadores filtrando por tópico.
Exploramos distintas temáticas disponibles (Health, Education, Poverty, Infrastructure, etc.).

In [ ]:
# primero obtenemos todos los tópicos disponibles via facets
payload_facets = {
    "count": True,
    "filter": "type eq 'indicator'",
    "select": "series_description/idno, series_description/name, series_description/topics",
    "facets": ["series_description/topics/name"],
    "top": 1,
    "skip": 0
}

resp_facets = post_api("/data360/searchv2", payload_facets)

# extraemos las facetas de tópicos
facets = resp_facets.get('@search.facets', {})
topics_facet = facets.get('series_description/topics/name', [])

if topics_facet:
    df_topics = pd.DataFrame(topics_facet)
    print(f"Tópicos disponibles ({len(df_topics)}):")
    display(df_topics.sort_values('count', ascending=False).reset_index(drop=True))
else:
    print("No se encontraron facetas de tópicos en la respuesta.")
    print("Claves disponibles en la respuesta:", list(resp_facets.keys()))

Tópicos disponibles (10):


,value,count
0,Prosperity,5608
1,Economic Policy,2471
2,Finance,1610
3,Macro-financial Policies,1561
4,Financial Stability and Integrity,1201
5,"Trade, Investment and Competitiveness",1064
6,People,881
7,Institutions,843
8,Fiscal Policy,686
9,Investment and Business Climate,630


Buscamos indicadores específicos por término de búsqueda y temática.

In [ ]:
# buscamos por palabra clave en el nombre del indicador
SEARCH_TERM = "poverty"   # "education", "health", "gdp"

payload_search = {
    "count": True,
    "filter": "type eq 'indicator'",
    "select": "series_description/idno, series_description/name, series_description/database_id, series_description/topics",
    "search": SEARCH_TERM,
    "top": 1000,
    "skip": 0
}

resp_search = post_api("/data360/searchv2", payload_search)

print(f"Búsqueda: '{SEARCH_TERM}'")
print(f"Total de resultados: {resp_search.get('@odata.count', 'N/A')}")

records_search = []
for item in resp_search.get('value', []):
    sd = item.get('series_description', {})
    topics = [t.get('name','') for t in sd.get('topics', [])]
    records_search.append({
        'indicator_id': sd.get('idno', ''),
        'nombre': sd.get('name', ''),
        'database_id': sd.get('database_id', ''),
        'tópicos': ', '.join(topics)
    })

df_search = pd.DataFrame(records_search)
display(df_search)

Búsqueda: 'poverty'
Total de resultados: 657


,indicator_id,nombre,database_id,tópicos
0,WB_SSGD_POVERTY_RATIO_NPL,Poverty headcount ratio at national poverty lines,WB_SSGD,"Prosperity, Poverty, Poverty"
1,WB_WDI_SI_POV_LMIC,Poverty headcount ratio at $4.20 a day (2021 PPP) (% of population),WB_WDI,"Prosperity, Poverty, Poverty"
2,WB_WDI_SI_POV_GAPS,Poverty gap at $3.00 a day (2021 PPP) (%),WB_WDI,"Prosperity, Poverty"
3,WB_WDI_SI_POV_NAHC,Poverty headcount ratio at national poverty lines (% of population),WB_WDI,"Prosperity, Poverty, Poverty, Trends in the Determinants of Food Security Ou..."
4,WB_WDI_SI_POV_UMIC,Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population),WB_WDI,"Prosperity, Poverty, Poverty"
...,...,...,...,...
652,WB_WDI_EN_URB_LCTY,Population in largest city,WB_WDI,"Planet, Water, Water and the Economy, Infrastructure, Urban, Resilience and ..."
653,WB_WDI_EN_URB_MCTY_TL_ZS,Population in urban agglomerations of more than 1 million (% of total popula...,WB_WDI,"Infrastructure, Urban, Resilience and Land, City Governance and Management a..."
654,WB_WDI_ER_H2O_FWDM_ZS,"Annual freshwater withdrawals, domestic (% of total freshwater withdrawal)",WB_WDI,
655,WB_WDI_ER_H2O_FWIN_ZS,"Annual freshwater withdrawals, industry (% of total freshwater withdrawal)",WB_WDI,


---
## 6. Metadata de un indicador
El endpoint `/data360/metadata` (POST) devuelve toda la información descriptiva de un indicador:
nombre, descripción, fuente, cobertura temporal, metodología, etc.

In [ ]:
INDICATOR_ID = "WB_WDI_SP_POP_TOTL"

payload_meta = {
    "query": f"&$filter=series_description/idno eq '{INDICATOR_ID}'"
}

resp_meta = post_api("/data360/metadata", payload_meta)

# segun como sea la estructura de la respuesta, extraemos los registros de metadata
if isinstance(resp_meta, dict):
    meta_values = resp_meta.get('value', [resp_meta])
elif isinstance(resp_meta, list):
    meta_values = resp_meta
else:
    meta_values = []

print(f"Registros de metadata para '{INDICATOR_ID}': {len(meta_values)}")

if meta_values:
    print("\nEstructura del primer registro (claves de primer nivel):")
    print(list(meta_values[0].keys()))

Registros de metadata para 'WB_WDI_SP_POP_TOTL': 1

Estructura del primer registro (claves de primer nivel):
['@search.score', 'id', 'idno', 'type', 'subtype', 'disaggregation_types', 'isDelete', 'cfPath', 'doc_type', 'remove_chart_type', 'data_confidentiality_code', 'data_confidentiality_name', 'dsd_name', 'dsd_version', 'dsd_codelist', 'is_active', 'metadata_information', 'series_description', 'tags', 'additional', 'product', 'disaggregation_codes', 'admin_metadata']


Extraemos los campos más relevantes de la metadata del indicador.

Nota: este ejemplo muestra la extracción **manual** con fines ilustrativos; más adelante encapsulamos esta lógica en funciones reutilizables (`get_indicator_meta_data` y `extract_meta_data_info`) para aplicarla de forma consistente a múltiples indicadores.

In [ ]:
if meta_values:
    meta = meta_values[0]
    sd = meta.get('series_description', {})

    campos_interes = [
        ('ID del indicador', sd.get('idno') or ''),
        ('Nombre', sd.get('name') or ''),
        ('Base de datos', f"{sd.get('database_name','')} ({sd.get('database_id','')})"),

        ('Descripción',
        sd.get('definition_short')
        or sd.get('definition_long')
        or ''),

        ('Unidad de medida', sd.get('measurement_unit') or ''),
        ('Periodicidad', sd.get('periodicity') or ''),

        ('Cobertura temporal',
        ' - '.join([
            sd.get('time_periods', [{}])[0].get('start', ''),
            sd.get('time_periods', [{}])[0].get('end', '')
        ]) if sd.get('time_periods') else ''),

        ('Tópicos',
        ', '.join(t.get('name', '') for t in sd.get('topics', []) if t.get('name'))),

        ('Fuentes',
        '; '.join(s.get('name', '') for s in sd.get('sources', []) if s.get('name'))),

        ('Entidad responsable',
        '; '.join(a.get('affiliation', '') for a in sd.get('authoring_entity', []) if a.get('affiliation'))),

        ('Cobertura geográfica',
        ', '.join(c.get('name', '') for c in sd.get('ref_country', [])[:5])),

        ('Metodología', sd.get('methodology') or ''),

        ('Limitaciones', sd.get('limitation') or ''),
    ]

    df_meta_display = pd.DataFrame(campos_interes, columns=['Campo', 'Valor'])
    display(df_meta_display)

    print("\nEstructura completa del campo 'series_description':")
    print(json.dumps(sd, indent=2, ensure_ascii=False)[:3000])

,Campo,Valor
0,ID del indicador,WB_WDI_SP_POP_TOTL
1,Nombre,"Population, total"
2,Base de datos,World Development Indicators (WDI) (WB_WDI)
3,Descripción,"Total population is based on the de facto definition of population, which co..."
4,Unidad de medida,Unit
5,Periodicidad,Annual
6,Cobertura temporal,1960 - 2024
7,Tópicos,
8,Fuentes,World Population Prospects; Statistical databases and publications from nati...
9,Entidad responsable,"United Nations Population Division, Eurostat, National Statistical Office"



Estructura completa del campo 'series_description':
{
  "idno": "WB_WDI_SP_POP_TOTL",
  "doi": null,
  "name": "Population, total",
  "display_name": null,
  "database_id": "WB_WDI",
  "database_name": "World Development Indicators (WDI)",
  "date_last_update": null,
  "date_released": null,
  "measurement_unit": "Unit",
  "release_calendar": null,
  "periodicity": "Annual",
  "base_period": null,
  "definition_short": null,
  "definition_long": "Total population is based on the de facto definition of population, which counts all residents regardless of legal status or citizenship. The values shown are midyear estimates.",
  "statistical_concept": "Estimates of total population describe the size of total population. Population estimates are dependent on the demographic components of change that are fertility, mortality and migration. As the size of population continues to change throughout the time even within a year, a reference time in the year that the estimate refers to is needed,

---
## 7. Desagregaciones disponibles para un indicador
El endpoint `/data360/disaggregation` muestra todas las dimensiones de corte disponibles:
sexo, edad, urbanización, breakdowns específicos, etc.

In [ ]:
INDICATOR_DISAG = "WB_WDI_SE_ADT_LITR_ZS"

resp_disag = get_api("/data360/disaggregation", params={
    "datasetId": DB_ID,
    "indicatorId": INDICATOR_DISAG
})

print(f"Desagregaciones para indicador '{INDICATOR_DISAG}':")
print(json.dumps(resp_disag, indent=2, ensure_ascii=False)[:3000])

Desagregaciones para indicador 'WB_WDI_SE_ADT_LITR_ZS':
[
  {
    "field_name": "FREQ",
    "label_name": "FREQ",
    "field_value": [
      "A"
    ]
  },
  {
    "field_name": "REF_AREA",
    "label_name": "REF_AREA",
    "field_value": [
      "ABW",
      "AFE",
      "AFG",
      "AFW",
      "AGO",
      "ALB",
      "ARB",
      "ARE",
      "ARG",
      "ARM",
      "ASM",
      "ATG",
      "AZE",
      "BDI",
      "BEN",
      "BFA",
      "BGD",
      "BGR",
      "BHR",
      "BIH",
      "BLR",
      "BLZ",
      "BOL",
      "BRA",
      "BRB",
      "BRN",
      "BTN",
      "BWA",
      "CAF",
      "CEB",
      "CHL",
      "CHN",
      "CIV",
      "CMR",
      "COD",
      "COG",
      "COL",
      "COM",
      "CPV",
      "CRI",
      "CSS",
      "CUB",
      "CYM",
      "CYP",
      "DOM",
      "DZA",
      "EAP",
      "EAR",
      "EAS",
      "ECA",
      "ECS",
      "ECU",
      "EGY",
      "ERI",
      "ESP",
      "EST",
      "ETH",
      "FCS",
     

Si el indicador anterior no tiene desagregaciones, probamos con uno conocido por tener cortes por sexo.

In [ ]:
INDICATOR_DISAG2 = "WB_WDI_SE_PRM_CMPT_FE_ZS"

resp_disag2 = get_api("/data360/disaggregation", params={
    "datasetId": DB_ID,
    "indicatorId": INDICATOR_DISAG2
})

print(f"Desagregaciones para '{INDICATOR_DISAG2}':")
if isinstance(resp_disag2, dict):
    disag_values = resp_disag2.get('value', [])
elif isinstance(resp_disag2, list):
    disag_values = resp_disag2
else:
    disag_values = []

if disag_values:
    df_disag = pd.DataFrame(disag_values)
    display(df_disag.head(30))
else:
    print(json.dumps(resp_disag2, indent=2, ensure_ascii=False)[:2000])

Desagregaciones para 'WB_WDI_SE_PRM_CMPT_FE_ZS':


,field_name,label_name,field_value
0,FREQ,FREQ,[A]
1,REF_AREA,REF_AREA,"[ABW, AFE, AFG, AFW, AGO, ALB, AND, ARB, ARE, ARG, ARM, ATG, AUT, AZE, BDI, ..."
2,INDICATOR,INDICATOR,[WB_WDI_SE_PRM_CMPT_FE_ZS]
3,SEX,SEX,[F]
4,AGE,AGE,[_T]
5,URBANISATION,URBANISATION,[_T]
6,UNIT_MEASURE,UNIT_MEASURE,[PT]
7,COMP_BREAKDOWN_1,COMP_BREAKDOWN_1,[_Z]
8,COMP_BREAKDOWN_2,COMP_BREAKDOWN_2,[_Z]
9,COMP_BREAKDOWN_3,COMP_BREAKDOWN_3,[_Z]


---
## 8. Recuperación de datos: estructura temporal y geográfica
El endpoint `/data360/data` es el núcleo de la API. Retorna observaciones con la siguiente estructura:
- **REF_AREA**: código de país o región
- **TIME_PERIOD**: año o período
- **INDICATOR**: ID del indicador
- **OBS_VALUE**: valor observado
- **SEX, AGE, URBANISATION**: dimensiones de desagregación
- **FREQ**: frecuencia (A=anual, Q=trimestral, M=mensual)

Esta función recupera todos los datos de un indicador específico, manejando paginación automáticamente. La definimos ahora pero la usaremos más adelante.

In [ ]:
def fetch_all_data(database_id, indicator_id):
    """Recupera todos los datos de un indicador paginando con el parámetro skip."""
    all_records = []
    skip = 0
    page_size = 1000
    total = None

    while True:
        params = {
            "DATABASE_ID": database_id,
            "INDICATOR": indicator_id,
            "skip": skip
        }
        resp = get_api("/data360/data", params=params)
        records = resp.get('value', [])

        if total is None:
            total = int(resp.get('count', 0))

        all_records.extend(records)
        print(f"  Pagina {skip // page_size + 1}: {len(records)} registros (skip={skip}, acumulado={len(all_records)}/{total})")

        if not records or len(all_records) >= total:
            break

        skip += page_size

    return all_records, total

Primero hacemos un GET para obtener los datos de una página de forma tal de poder entender la estructura de la respuesta y los campos disponibles.

In [ ]:
# traemos datos del indicador de población total para todos los países
params_data = {
    "DATABASE_ID": "WB_WDI",
    "INDICATOR": "WB_WDI_SP_POP_TOTL",
    "skip": 0
}

resp_data = get_api("/data360/data", params=params_data)

total_records = resp_data.get('count', 0)
records_page  = resp_data.get('value', [])

print(f"Total de registros disponibles: {total_records}")
print(f"Registros en esta página (máx 1000): {len(records_page)}")
print(f"\nColumnas de cada registro:")
if records_page:
    print(list(records_page[0].keys()))

Total de registros disponibles: 17195
Registros en esta página (máx 1000): 1000

Columnas de cada registro:
['OBS_VALUE', 'TIME_FORMAT', 'UNIT_MULT', 'COMMENT_OBS', 'OBS_STATUS', 'OBS_CONF', 'AGG_METHOD', 'DECIMALS', 'COMMENT_TS', 'DATA_SOURCE', 'LATEST_DATA', 'DATABASE_ID', 'INDICATOR', 'REF_AREA', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3', 'TIME_PERIOD', 'FREQ', 'UNIT_MEASURE', 'UNIT_TYPE']


Convertimos los datos en un DataFrame y exploramos su estructura.

In [ ]:
df_data = pd.DataFrame(records_page)

df_data['OBS_VALUE']   = pd.to_numeric(df_data['OBS_VALUE'], errors='coerce')
df_data['TIME_PERIOD'] = pd.to_numeric(df_data['TIME_PERIOD'], errors='coerce')

print(f"Shape: {df_data.shape}")
print(f"\nColumnas y tipos de dato:")
print(df_data.dtypes)
print(f"\nEjemplo de registros:")
display(df_data.head(10))

Shape: (1000, 24)

Columnas y tipos de dato:
OBS_VALUE            int64
TIME_FORMAT         object
UNIT_MULT            int64
COMMENT_OBS         object
OBS_STATUS          object
OBS_CONF            object
AGG_METHOD          object
DECIMALS            object
COMMENT_TS          object
DATA_SOURCE         object
LATEST_DATA           bool
DATABASE_ID         object
INDICATOR           object
REF_AREA            object
SEX                 object
AGE                 object
URBANISATION        object
COMP_BREAKDOWN_1    object
COMP_BREAKDOWN_2    object
COMP_BREAKDOWN_3    object
TIME_PERIOD          int64
FREQ                object
UNIT_MEASURE        object
UNIT_TYPE           object
dtype: object

Ejemplo de registros:


,OBS_VALUE,TIME_FORMAT,UNIT_MULT,COMMENT_OBS,OBS_STATUS,OBS_CONF,AGG_METHOD,DECIMALS,COMMENT_TS,DATA_SOURCE,LATEST_DATA,DATABASE_ID,INDICATOR,REF_AREA,SEX,AGE,URBANISATION,COMP_BREAKDOWN_1,COMP_BREAKDOWN_2,COMP_BREAKDOWN_3,TIME_PERIOD,FREQ,UNIT_MEASURE,UNIT_TYPE
0,28813101,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,VEN,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
1,87455152,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,VNM,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
2,108357,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,VIR,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
3,3786161,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,PSE,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
4,26754387,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,YEM,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
5,13965594,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,ZMB,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
6,13356548,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,ZWE,_T,_T,_T,_Z,_Z,_Z,2010,A,PS,None
7,544737983,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,AFE,_T,_T,_T,_Z,_Z,_Z,2011,A,PS,None
8,374790143,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,AFW,_T,_T,_T,_Z,_Z,_Z,2011,A,PS,None
9,372054934,P1Y,0,None,A,PU,_Z,2,"Population, total",WB_WDI,False,WB_WDI,WB_WDI_SP_POP_TOTL,ARB,_T,_T,_T,_Z,_Z,_Z,2011,A,PS,None


---
## 9. Exploración de la dimensión temporal
Analizamos el rango de años y la cobertura temporal de los datos disponibles.

In [ ]:
print("=== Dimensión TEMPORAL ===")
print(f"Años disponibles en esta página:")
print(sorted(df_data['TIME_PERIOD'].dropna().unique().astype(int).tolist()))
print(f"\nRango: {df_data['TIME_PERIOD'].min():.0f} - {df_data['TIME_PERIOD'].max():.0f}")
print(f"\nFrecuencias (FREQ) disponibles:")
print(df_data['FREQ'].value_counts())
print("\nFormatos de tiempo (TIME_FORMAT):")
print(df_data['TIME_FORMAT'].value_counts())

=== Dimensión TEMPORAL ===
Años disponibles en esta página:
[2000, 2003, 2004, 2005, 2006, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

Rango: 2000 - 2022

Frecuencias (FREQ) disponibles:
FREQ
A    1000
Name: count, dtype: int64

Formatos de tiempo (TIME_FORMAT):
TIME_FORMAT
P1Y    1000
Name: count, dtype: int64


---
## 10. Exploración de la dimensión geográfica
Inspeccionamos los códigos de área de referencia (países, regiones, grupos) presentes en los datos.

In [ ]:
print("=== Dimensión GEOGRÁFICA (REF_AREA) ===")
areas = df_data['REF_AREA'].value_counts()
print(f"Total de áreas/países únicos en esta página: {len(areas)}")
print("\nÁreas con más registros:")
print(areas.head(30).to_string())

=== Dimensión GEOGRÁFICA (REF_AREA) ===
Total de áreas/países únicos en esta página: 233

Áreas con más registros:
REF_AREA
IRQ    10
VNM     8
VIR     8
PSE     8
YEM     8
SYR     8
TJK     8
CHE     8
VEN     8
IRN     8
IDN     8
IND     8
SUR     7
OMN     7
TZA     7
ZWE     7
ZMB     7
THA     7
IMN     7
IRL     7
ISL     7
SWE     7
SDN     7
VUT     7
UZB     7
FRA     7
BWA     6
TLS     6
MNP     6
ISR     6


---
## 11. Exploración de dimensiones de desagregación
Analizamos las dimensiones SEX, AGE, URBANISATION y COMP_BREAKDOWN para entender qué cortes existen.

In [ ]:
dimensiones = ['SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3']

for dim in dimensiones:
    if dim in df_data.columns:
        valores = df_data[dim].value_counts()
        print(f"\n--- {dim} ---")
        print(valores.to_string())


--- SEX ---
SEX
_T    1000

--- AGE ---
AGE
_T    1000

--- URBANISATION ---
URBANISATION
_T    1000

--- COMP_BREAKDOWN_1 ---
COMP_BREAKDOWN_1
_Z    1000

--- COMP_BREAKDOWN_2 ---
COMP_BREAKDOWN_2
_Z    1000

--- COMP_BREAKDOWN_3 ---
COMP_BREAKDOWN_3
_Z    1000


---
## 12. Paginación: obtener todos los registros
La API devuelve máximo 1000 registros por llamada. Entonces, vamos a usar la función definida anteriormente.

In [ ]:
print("Descargando todos los datos del indicador WB_WDI_SP_POP_TOTL...")
all_records, total_api = fetch_all_data("WB_WDI", "WB_WDI_SP_POP_TOTL")

df_full = pd.DataFrame(all_records)
df_full['OBS_VALUE']   = pd.to_numeric(df_full['OBS_VALUE'], errors='coerce')
df_full['TIME_PERIOD'] = pd.to_numeric(df_full['TIME_PERIOD'], errors='coerce')

print(f"\nTotal descargado: {len(df_full)} de {total_api} registros")

Descargando todos los datos del indicador WB_WDI_SP_POP_TOTL...
  Pagina 1: 1000 registros (skip=0, acumulado=1000/17195)
  Pagina 2: 1000 registros (skip=1000, acumulado=2000/17195)
  Pagina 3: 1000 registros (skip=2000, acumulado=3000/17195)
  Pagina 4: 1000 registros (skip=3000, acumulado=4000/17195)
  Pagina 5: 1000 registros (skip=4000, acumulado=5000/17195)
  Pagina 6: 1000 registros (skip=5000, acumulado=6000/17195)
  Pagina 7: 1000 registros (skip=6000, acumulado=7000/17195)
  Pagina 8: 1000 registros (skip=7000, acumulado=8000/17195)
  Pagina 9: 1000 registros (skip=8000, acumulado=9000/17195)
  Pagina 10: 1000 registros (skip=9000, acumulado=10000/17195)
  Pagina 11: 1000 registros (skip=10000, acumulado=11000/17195)
  Pagina 12: 1000 registros (skip=11000, acumulado=12000/17195)
  Pagina 13: 1000 registros (skip=12000, acumulado=13000/17195)
  Pagina 14: 1000 registros (skip=13000, acumulado=14000/17195)
  Pagina 15: 1000 registros (skip=14000, acumulado=15000/17195)
  Pagin

---
## 13. Filtrado por país y rango temporal
Traemos datos para Argentina en un rango de años.

In [ ]:
# Datos de Argentina (ARG) entre 2000 y 2023
params_arg = {
    "DATABASE_ID": "WB_WDI",
    "INDICATOR": "WB_WDI_SP_POP_TOTL",
    "REF_AREA": "ARG",
    "timePeriodFrom": "2000",
    "timePeriodTo": "2023",
    "skip": 0
}

resp_arg = get_api("/data360/data", params=params_arg)
df_arg = pd.DataFrame(resp_arg.get('value', []))
df_arg['OBS_VALUE']   = pd.to_numeric(df_arg['OBS_VALUE'], errors='coerce')
df_arg['TIME_PERIOD'] = pd.to_numeric(df_arg['TIME_PERIOD'], errors='coerce')

print(f"Registros para ARG (2000-2023): {len(df_arg)}")
display(df_arg[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'UNIT_MEASURE', 'FREQ']].sort_values('TIME_PERIOD'))

Registros para ARG (2000-2023): 24


,REF_AREA,TIME_PERIOD,OBS_VALUE,UNIT_MEASURE,FREQ
4,ARG,2000,37213984,PS,A
18,ARG,2001,37624825,PS,A
6,ARG,2002,38029349,PS,A
0,ARG,2003,38424282,PS,A
5,ARG,2004,38815916,PS,A
12,ARG,2005,39216789,PS,A
17,ARG,2006,39622115,PS,A
10,ARG,2007,40016763,PS,A
19,ARG,2008,40424148,PS,A
3,ARG,2009,40854831,PS,A


In [ ]:
if not df_arg.empty and 'TIME_PERIOD' in df_arg.columns:

    df_arg_chart = (
        df_arg[['TIME_PERIOD', 'OBS_VALUE']]
        .dropna()
        .sort_values('TIME_PERIOD')
        .assign(Poblacion=lambda d: d['OBS_VALUE'] / 1e6)
    )

    chart_arg = (
        alt.Chart(df_arg_chart)
        .mark_line(point=True)
        .encode(
            x=alt.X('TIME_PERIOD:O', title='Año', axis=alt.Axis(labelAngle=0)),
            y=alt.Y('Poblacion:Q', title='Población (millones)'),
            tooltip=[
                alt.Tooltip('TIME_PERIOD:O', title='Año'),
                alt.Tooltip('Poblacion:Q', title='Población (millones)', format='.2f')
            ]
        )
        .properties(
            title='Evolución de la población total de Argentina (2000-2023)',
            width=800,
            height=350
        )
        .interactive()
    )

    display(chart_arg)
else:
    print('No hay datos de Argentina para graficar.')

alt.Chart(...)

---
## 14. Exploración de múltiples bases de datos
Iteramos sobre distintas bases de datos del Banco Mundial para comparar su cantidad de indicadores.

In [ ]:
# tomamos hasta 10 bases de datos para no saturar la API
databases_sample = df_databases['database_id'].dropna().head(10).tolist()

resumen_dbs = []
for db_id in databases_sample:
    try:
        resp = get_api("/data360/indicators", params={"datasetId": db_id})
        if isinstance(resp, list):
            n_indicators = len(resp)
        elif isinstance(resp, dict):
            n_indicators = len(resp.get('value', []))
        else:
            n_indicators = 0
        nombre = df_databases.loc[df_databases['database_id'] == db_id, 'nombre'].values
        nombre = nombre[0] if len(nombre) > 0 else db_id
        resumen_dbs.append({'database_id': db_id, 'nombre': nombre, 'n_indicadores': n_indicators})
        print(f"  {db_id}: {n_indicators} indicadores")
    except Exception as e:
        print(f"  {db_id}: Error - {e}")

df_resumen = pd.DataFrame(resumen_dbs).sort_values('n_indicadores', ascending=False)
display(df_resumen)

  IMF_IRFCL: 120 indicadores
  FH_NIT: 11 indicadores
  IBP_OBS: 6 indicadores
  ILO_EMP: 1 indicadores
  WB_WITS: 44 indicadores
  FAO_AS: 124 indicadores
  WB_IDS: 102 indicadores
  WB_SHP: 3 indicadores
  WB_EWSA: 29 indicadores
  JRC_EDGAR: 10 indicadores


,database_id,nombre,n_indicadores
5,FAO_AS,Aquastat,124
0,IMF_IRFCL,International Reserves and Foreign Currency Liquidity (IRFCL),120
6,WB_IDS,International Debt Statistics (IDS),102
4,WB_WITS,World Integrated Trade Solution (WITS),44
8,WB_EWSA,Europe and Central Asia (ECA) Water Security Assessment,29
1,FH_NIT,Nations in Transit,11
9,JRC_EDGAR,Emissions Database for Global Atmospheric Research (EDGAR),10
2,IBP_OBS,Open Budget Survey,6
7,WB_SHP,Global Database Of Shared Prosperity,3
3,ILO_EMP,Employment by economic activity,1


In [ ]:
if not df_resumen.empty and df_resumen['n_indicadores'].sum() > 0:
    df_plot = (
        df_resumen[df_resumen['n_indicadores'] > 0]
        .sort_values('n_indicadores', ascending=False)
    )

    chart_indicadores = (
        alt.Chart(df_plot)
        .mark_bar()
        .encode(
            x=alt.X('n_indicadores:Q', title='Numero de indicadores'),
            y=alt.Y('database_id:N', sort='-x', title='Base de datos'),
            tooltip=[
                alt.Tooltip('database_id:N', title='Base de datos'),
                alt.Tooltip('nombre:N', title='Nombre'),
                alt.Tooltip('n_indicadores:Q', title='Indicadores')
            ]
        )
        .properties(
            title='Cantidad de indicadores por base de datos (muestra)',
            width=800,
            height=350
        )
        .interactive()
    )

    display(chart_indicadores)

alt.Chart(...)

---
## 15. Conclusiones: estructura de la API Data360

A partir de la exploración realizada, podemos concluir:

### Bases de datos
- La API contiene múltiples bases de datos identificadas por `DATABASE_ID` (ej: `WB_WDI`, `WB_GEM`, etc.)
- Cada base de datos agrupa indicadores relacionados por temática o fuente

### Indicadores
- Cada indicador tiene un ID único con prefijo del database (ej: `WB_WDI_SP_POP_TOTL`)
- Hay metadatos ricos: descripción, fuente, unidad, cobertura, tópicos
- La búsqueda textual y por tópicos facilita descubrir indicadores relevantes

### Estructura de los datos
| Dimensión       | Campo API       | Descripción |
|-----------------|-----------------|-------------|
| Geográfica      | `REF_AREA`      | Código ISO3 de país, región o grupo |
| Temporal        | `TIME_PERIOD`   | Año o período (YYYY, YYYY-Qn, etc.) |
| Temática        | `INDICATOR`     | ID del indicador |
| Sexo            | `SEX`           | F / M / _T (total) |
| Edad            | `AGE`           | Grupo etario o _T |
| Urbanización    | `URBANISATION`  | Urbano / Rural / _T |
| Desagregación   | `COMP_BREAKDOWN_1/2/3` | Dimensiones adicionales |
| Frecuencia      | `FREQ`          | A=anual, Q=trimestral, M=mensual |
| Unidad          | `UNIT_MEASURE`  | PT=%, USD, personas, etc. |

**Las visualizaciones anteriores sobre población total se utilizan como ejemplo exploratorio para comprender la estructura de los datos, la cobertura temporal y geográfica.**
**También permiten ilustrar el funcionamiento del endpoint ` /data360/data `.**

<br>

**A continuación, se desarrolla el caso de estudio principal del trabajo, centrado en la preparación estructural de los países para aprovechar tecnologías basadas en inteligencia artificial.**

## 16. Selección del tema: preparación para la IA en la región sudamericana

Para este trabajo, proponemos estudiar qué tan preparados están distintos países
de Sudamérica para aprovechar tecnologías basadas en inteligencia artificial.

Dado que la "preparación para la IA" no se observa directamente en un único
indicador, la aproximamos mediante seis dimensiones:

1. Conectividad e infraestructura digital
2. Capital humano
3. Inversión en I+D
4. Capacidad investigadora
5. Integración tecnológica
6. Capacidad económica base

En esta sección exploramos indicadores candidatos dentro de la API Data360 para
construir una aproximación comparativa entre países de la región.

In [ ]:
search_terms = [
    "internet",
    "broadband",
    "literacy",
    "education",
    "school enrollment",
    "gdp per capita",
    "technology",
    "research"
]

resultados = []

for term in search_terms:
    payload_search = {
        "count": True,
        "filter": "type eq 'indicator'",
        "select": "series_description/idno, series_description/name, series_description/database_id, series_description/topics",
        "search": term,
        "top": 20,
        "skip": 0
    }

    resp = post_api("/data360/searchv2", payload_search)

    for item in resp.get("value", []):
        sd = item.get("series_description", {})
        resultados.append({
            "search_term": term,
            "indicator_id": sd.get("idno", ""),
            "nombre": sd.get("name", ""),
            "database_id": sd.get("database_id", ""),
            "topicos": ", ".join(t.get("name", "") for t in sd.get("topics", []))
        })

df_candidates = pd.DataFrame(resultados)


In [ ]:
display(df_candidates)

,search_term,indicator_id,nombre,database_id,topicos
0,internet,WB_SSGD_PCT_POP_INTERNET_USAGE,Percentage of population who use the internet,WB_SSGD,
1,internet,WB_SSGD_PCT_HHS_INTERNET,Percentage of households that have internet,WB_SSGD,
2,internet,WB_SSGD_PCT_INTERNET_USERS,Individuals using the Internet (% of population),WB_SSGD,
3,internet,WB_WDI_IT_NET_USER_ZS,Individuals using the Internet (% of population),WB_WDI,"Digital, Digital Services, Connectivity"
4,internet,UNCTAD_DE_ICT_CORE_UISIC4_ANN_00_B8,Proportion of businesses placing orders over the Internet,UNCTAD_DE,"Digital, Connectivity, Digital Adoption, Digital Transformation Database, Di..."
...,...,...,...,...,...
155,research,JRC_EDGAR_HG,Total mercury emissions,JRC_EDGAR,"Planet, Climate Change"
156,research,JRC_EDGAR_NMVOC,Non-Methane Volatile Organic Compounds emissions,JRC_EDGAR,"Planet, Climate Change"
157,research,JRC_EDGAR_NOX,Nitrogen Oxides emissions,JRC_EDGAR,"Planet, Climate Change"
158,research,JRC_EDGAR_OC,Carbonaceous speciation emissions (OC),JRC_EDGAR,"Planet, Climate Change"


## 17. Selección final de indicadores

A partir de la exploración de indicadores candidatos dentro de la API Data360, se seleccionaron seis variables para construir el caso de estudio sobre preparación estructural para la IA en países de América Latina.
La elección combinó pertinencia conceptual y disponibilidad razonable de datos.

Los indicadores seleccionados fueron:

| # | Indicador | Dimensión |
|---|-----------|-----------|
| 1 | `WB_WDI_SE_TER_ENRR` | Capital humano |
| 2 | `WB_WDI_IT_NET_USER_ZS` | Conectividad |
| 3 | `WB_WDI_GB_XPD_RSDV_GD_ZS` | Inversión en I+D |
| 4 | `WB_WDI_SP_POP_SCIE_RD_P6` | Capacidad investigadora |
| 5 | `WB_WDI_TX_VAL_TECH_MF_ZS` | Integración tecnológica |
| 6 | `WB_WDI_NY_GDP_PCAP_KD` | Capacidad económica base |


La decisión final sobre el indicador educativo se justificará en la sección siguiente, a partir de la comparación de cobertura temporal y geográfica frente a otras alternativas inicialmente consideradas.

In [ ]:
DB_ID_CASE = "WB_WDI"

LATAM = ["ARG", "BRA", "CHL", "COL", "MEX", "PER", "URY"]

candidate_indicators = {
    # Condiciones de adopción (qué tan listo está el país para usar IA)
    "gdp_per_capita":      "WB_WDI_NY_GDP_PCAP_KD",
    "internet_usage":      "WB_WDI_IT_NET_USER_ZS",
    "tertiary_enrollment": "WB_WDI_SE_TER_ENRR",

    # Capacidad de desarrollo (qué tan listo está para producir IA)
    "rd_expenditure":      "WB_WDI_GB_XPD_RSDV_GD_ZS",
    "researchers":         "WB_WDI_SP_POP_SCIE_RD_P6",
    "hightech_exports":    "WB_WDI_TX_VAL_TECH_MF_ZS",
}

candidate_indicators

{'gdp_per_capita': 'WB_WDI_NY_GDP_PCAP_KD',
 'internet_usage': 'WB_WDI_IT_NET_USER_ZS',
 'tertiary_enrollment': 'WB_WDI_SE_TER_ENRR',
 'rd_expenditure': 'WB_WDI_GB_XPD_RSDV_GD_ZS',
 'researchers': 'WB_WDI_SP_POP_SCIE_RD_P6',
 'hightech_exports': 'WB_WDI_TX_VAL_TECH_MF_ZS'}

## 18. Metadata de los indicadores

Encapsulamos la lógica anterior en funciones reutilizables para consultar la metadata y extraer campos clave de cada indicador.

In [ ]:
df_indicators = pd.DataFrame({"indicator_id": list(candidate_indicators.values())})

In [ ]:
def get_indicator_meta_data(INDICATOR_ID):
    """Retorna el campo values de los metadatos para un determinado INDICATOR_ID."""
    payload_meta = {
        "query": f"&$filter=series_description/idno eq '{INDICATOR_ID}'"
    }
    resp_meta = post_api("/data360/metadata", payload_meta)

    if isinstance(resp_meta, dict):
        meta_values = resp_meta.get('value', [resp_meta])
    elif isinstance(resp_meta, list):
        meta_values = resp_meta
    else:
        meta_values = []

    return meta_values

In [ ]:
def extract_meta_data_info(meta_values):
    """Extrae de los metadatos los campos de interés."""
    if meta_values:
        meta = meta_values[0]
        sd = meta.get('series_description', {})

        campos_interes = [
            ('ID del indicador', sd.get('idno') or ''),
            ('Nombre', sd.get('name') or ''),
            ('Base de datos', f"{sd.get('database_name','')} ({sd.get('database_id','')})" ),
            ('Descripción',
             sd.get('definition_short')
             or sd.get('definition_long')
             or ''),
            ('Unidad de medida', sd.get('measurement_unit') or ''),
            ('Periodicidad', sd.get('periodicity') or ''),
            ('Cobertura temporal',
             ' - '.join([
                 sd.get('time_periods', [{}])[0].get('start', ''),
                 sd.get('time_periods', [{}])[0].get('end', '')
             ]) if sd.get('time_periods') else ''),
            ('Tópicos',
             ', '.join(t.get('name', '') for t in sd.get('topics', []) if t.get('name'))),
            ('Fuentes',
             '; '.join(s.get('name', '') for s in sd.get('sources', []) if s.get('name'))),
            ('Entidad responsable',
             '; '.join(a.get('affiliation', '') for a in sd.get('authoring_entity', []) if a.get('affiliation'))),
            ('Cobertura geográfica',
             ', '.join(c.get('name', '') for c in sd.get('ref_country', [])[:5])),
            ('Metodología', sd.get('methodology') or ''),
            ('Limitaciones', sd.get('limitation') or ''),
        ]
        return sd, campos_interes
    else:
        return None, None

In [ ]:
def build_metadata_dataframe(df_indicators):
    """Construye un dataframe consolidado con la metadata de todos los indicadores."""
    metadata_rows = []

    for ind in df_indicators["indicator_id"]:
        try:
            meta = get_indicator_meta_data(ind)
            sd, campos_interes = extract_meta_data_info(meta)
            meta_dict = dict(campos_interes)
            meta_dict["indicator_id"] = ind
            metadata_rows.append(meta_dict)
        except Exception as e:
            print(f"⚠️ Error con {ind}: {e}")

    return pd.DataFrame(metadata_rows)

In [ ]:
df_metadata = build_metadata_dataframe(df_indicators)
display(df_metadata[['ID del indicador', 'Nombre', 'Descripción', 'Unidad de medida']])

,ID del indicador,Nombre,Descripción,Unidad de medida
0,WB_WDI_NY_GDP_PCAP_KD,GDP per capita (constant 2015 US$),Gross domestic product is the total income earned through the production of ...,constant 2015 US$
1,WB_WDI_IT_NET_USER_ZS,Individuals using the Internet (% of population),Internet users are individuals who have used the Internet (from any location...,% of population
2,WB_WDI_SE_TER_ENRR,"School enrollment, tertiary (% gross)","Gross enrollment ratio is the ratio of total enrollment, regardless of age, ...",% of population in the 5-year age group immediately following upper secondar...
3,WB_WDI_GB_XPD_RSDV_GD_ZS,Research and development expenditure (% of GDP),"Gross domestic expenditures on research and development (R&D), expressed as ...",% of GDP
4,WB_WDI_SP_POP_SCIE_RD_P6,Researchers in R&D (per million people),"The number of researchers engaged in Research &Development (R&D), expressed ...",Per million people
5,WB_WDI_TX_VAL_TECH_MF_ZS,High-technology exports (% of manufactured exports),"High-technology exports are products with high R&D intensity, such as in aer...",% of GDP


## 19. Descarga de datos para el caso de estudio

In [ ]:
def fetch_country_indicator(database_id, indicator_id, country, time_from=2000, time_to=2023):
    params = {
        "DATABASE_ID": database_id,
        "INDICATOR": indicator_id,
        "REF_AREA": country,
        "timePeriodFrom": str(time_from),
        "timePeriodTo": str(time_to),
        "skip": 0
    }

    resp = get_api("/data360/data", params=params)
    df = pd.DataFrame(resp.get("value", []))

    if not df.empty:
        df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")
        df["TIME_PERIOD"] = pd.to_numeric(df["TIME_PERIOD"], errors="coerce")

    return df

In [ ]:
all_case_data = []

for indicator_name, indicator_id in candidate_indicators.items():
    for country in LATAM:
        df_tmp = fetch_country_indicator(DB_ID_CASE, indicator_id, country, 2000, 2023)

        if not df_tmp.empty:
            df_tmp["indicator_name"] = indicator_name
            all_case_data.append(df_tmp)

df_case = pd.concat(all_case_data, ignore_index=True) if all_case_data else pd.DataFrame()

print(df_case.shape)

(831, 25)


In [ ]:
display(df_case)

,OBS_VALUE,TIME_FORMAT,UNIT_MULT,COMMENT_OBS,OBS_STATUS,OBS_CONF,AGG_METHOD,DECIMALS,COMMENT_TS,DATA_SOURCE,LATEST_DATA,DATABASE_ID,INDICATOR,REF_AREA,SEX,AGE,URBANISATION,COMP_BREAKDOWN_1,COMP_BREAKDOWN_2,COMP_BREAKDOWN_3,TIME_PERIOD,FREQ,UNIT_MEASURE,UNIT_TYPE,indicator_name
0,12285.378465,P1Y,0,None,A,PU,_Z,2,GDP per capita (constant 2015 US$),WB_WDI,False,WB_WDI,WB_WDI_NY_GDP_PCAP_KD,ARG,_T,_T,_T,_Z,_Z,_Z,2009,A,USD_K_2015,CUR,gdp_per_capita
1,13520.112985,P1Y,0,None,A,PU,_Z,2,GDP per capita (constant 2015 US$),WB_WDI,False,WB_WDI,WB_WDI_NY_GDP_PCAP_KD,ARG,_T,_T,_T,_Z,_Z,_Z,2017,A,USD_K_2015,CUR,gdp_per_capita
2,9545.531941,P1Y,0,None,A,PU,_Z,2,GDP per capita (constant 2015 US$),WB_WDI,False,WB_WDI,WB_WDI_NY_GDP_PCAP_KD,ARG,_T,_T,_T,_Z,_Z,_Z,2003,A,USD_K_2015,CUR,gdp_per_capita
3,11870.279156,P1Y,0,None,A,PU,_Z,2,GDP per capita (constant 2015 US$),WB_WDI,False,WB_WDI,WB_WDI_NY_GDP_PCAP_KD,ARG,_T,_T,_T,_Z,_Z,_Z,2006,A,USD_K_2015,CUR,gdp_per_capita
4,13197.357027,P1Y,0,None,A,PU,_Z,2,GDP per capita (constant 2015 US$),WB_WDI,False,WB_WDI,WB_WDI_NY_GDP_PCAP_KD,ARG,_T,_T,_T,_Z,_Z,_Z,2008,A,USD_K_2015,CUR,gdp_per_capita
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
826,9.903989,P1Y,0,None,A,PU,_Z,2,High-technology exports (% of manufactured exports),WB_WDI,False,WB_WDI,WB_WDI_TX_VAL_TECH_MF_ZS,URY,_T,_T,_T,_Z,_Z,_Z,2012,A,PT,RATIO,hightech_exports
827,7.636704,P1Y,0,None,A,PU,_Z,2,High-technology exports (% of manufactured exports),WB_WDI,False,WB_WDI,WB_WDI_TX_VAL_TECH_MF_ZS,URY,_T,_T,_T,_Z,_Z,_Z,2018,A,PT,RATIO,hightech_exports
828,9.204666,P1Y,0,None,A,PU,_Z,2,High-technology exports (% of manufactured exports),WB_WDI,False,WB_WDI,WB_WDI_TX_VAL_TECH_MF_ZS,URY,_T,_T,_T,_Z,_Z,_Z,2013,A,PT,RATIO,hightech_exports
829,7.260327,P1Y,0,None,A,PU,_Z,2,High-technology exports (% of manufactured exports),WB_WDI,False,WB_WDI,WB_WDI_TX_VAL_TECH_MF_ZS,URY,_T,_T,_T,_Z,_Z,_Z,2010,A,PT,RATIO,hightech_exports


## 20. Matriz de Correlación y Heatmap

In [ ]:
# --- 1. Tomar el valor más reciente por país e indicador ---
df_latest = (
    df_case[["REF_AREA", "indicator_name", "OBS_VALUE", "TIME_PERIOD"]]
    .dropna(subset=["OBS_VALUE"])
    .sort_values("TIME_PERIOD", ascending=False)
    .drop_duplicates(subset=["REF_AREA", "indicator_name"])
)

# --- 2. Pivot: filas = países, columnas = indicadores ---
df_pivot = (
    df_latest
    .pivot(index="REF_AREA", columns="indicator_name", values="OBS_VALUE")
    .dropna() # solo países con todos los indicadores disponibles
)

# --- 3. Matriz de correlación ---
df_corr = df_pivot.corr(method="pearson").round(2)

# --- 4. Pasar a formato largo para Altair ---
df_corr_long = (
    df_corr
    .reset_index()
    .melt(id_vars="indicator_name", var_name="indicator_name_2", value_name="correlation")
)

# --- 5. Heatmap ---
base = alt.Chart(df_corr_long).encode(
    x=alt.X("indicator_name:N",   title=None, axis=alt.Axis(labelAngle=-35)),
    y=alt.Y("indicator_name_2:N", title=None),
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        "correlation:Q",
        scale=alt.Scale(domain=[-1, 0, 1], scheme="redblue", reverse=True),
        title="Correlación"
    ),
    tooltip=[
        alt.Tooltip("indicator_name:N",   title="Indicador X"),
        alt.Tooltip("indicator_name_2:N", title="Indicador Y"),
        alt.Tooltip("correlation:Q",      title="Correlación", format=".2f"),
    ]
)

text = base.mark_text(fontSize=12, fontWeight="bold").encode(
    text=alt.Text("correlation:Q", format=".2f"),
    color=alt.condition(
        "abs(datum.correlation) > 0.6",
        alt.value("white"),
        alt.value("#444444")
    )
)

chart_corr = (
    (heatmap + text)
    .properties(
        title="Matriz de correlación entre indicadores",
        width=480,
        height=420
    )
    .configure_axis(labelFontSize=11)
    .configure_title(fontSize=14, anchor="start")
)

display(chart_corr)

alt.LayerChart(...)

## 21. Decisión metodológica final

A partir de la exploración de indicadores candidatos en la API Data360 y de la cobertura disponible para los países seleccionados, se definió un set de **seis** indicadores agrupados en dos dimensiones complementarias para aproximar la “preparación estructural” para la IA:

- **Adopción** — condiciones para incorporar y usar IA:
  - capacidad económica: `WB_WDI_NY_GDP_PCAP_KD` (`gdp_per_capita`)
  - conectividad: `WB_WDI_IT_NET_USER_ZS` (`internet_usage`)
  - capital humano (formación terciaria): `WB_WDI_SE_TER_ENRR` (`tertiary_enrollment`)

- **Desarrollo** — capacidades para producir e innovar en IA:
  - inversión en I+D: `WB_WDI_GB_XPD_RSDV_GD_ZS` (`rd_expenditure`)
  - masa crítica investigadora: `WB_WDI_SP_POP_SCIE_RD_P6` (`researchers`)
  - integración/sofisticación tecnológica: `WB_WDI_TX_VAL_TECH_MF_ZS` (`hightech_exports`)

En etapas preliminares se consideraron alternativas educativas (por ejemplo, alfabetización adulta o matrícula neta en secundaria), pero para el caso de estudio final se utiliza `WB_WDI_SE_TER_ENRR` por su mejor alineación conceptual con la formación de capital humano avanzado y una cobertura más adecuada para el recorte seleccionado.

Dado que la cobertura temporal no es idéntica entre indicadores, según el objetivo de cada visualización se emplean estrategias distintas: series temporales para observar evolución, comparación del último año disponible para estudiar relaciones entre variables y un índice exploratorio construido con el valor más reciente disponible por país.

## 22. Preparación del dataset analítico

In [ ]:
df_case_clean = df_case.copy()

df_case_clean = df_case_clean[
    ["REF_AREA", "TIME_PERIOD", "OBS_VALUE", "INDICATOR", "indicator_name", "UNIT_MEASURE"]
].dropna(subset=["OBS_VALUE", "TIME_PERIOD"])

df_case_clean["TIME_PERIOD"] = df_case_clean["TIME_PERIOD"].astype(int)

country_names = {
    "ARG": "Argentina",
    "BRA": "Brazil",
    "CHL": "Chile",
    "COL": "Colombia",
    "MEX": "Mexico",
    "PER": "Peru",
    "URY": "Uruguay"
}

df_case_clean["country"] = df_case_clean["REF_AREA"].map(country_names)

# Orden consistente de países para gráficos (leyendas/ejes)
country_order = ["Argentina", "Brazil", "Chile", "Colombia", "Mexico", "Peru", "Uruguay"]

print(df_case_clean.shape)
display(df_case_clean.head(10))

(831, 7)


,REF_AREA,TIME_PERIOD,OBS_VALUE,INDICATOR,indicator_name,UNIT_MEASURE,country
0,ARG,2009,12285.378465,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
1,ARG,2017,13520.112985,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
2,ARG,2003,9545.531941,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
3,ARG,2006,11870.279156,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
4,ARG,2008,13197.357027,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
5,ARG,2021,12549.281170,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
6,ARG,2011,14040.618912,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
7,ARG,2012,13754.425424,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
8,ARG,2015,13679.626498,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina
9,ARG,2023,12993.092887,WB_WDI_NY_GDP_PCAP_KD,gdp_per_capita,USD_K_2015,Argentina


## 23. Generamos índices

In [ ]:
# --- Pivot: valor más reciente por país e indicador ---
df_wide = (
    df_case_clean
    .sort_values("TIME_PERIOD", ascending=False)           # más reciente primero
    .drop_duplicates(subset=["country", "indicator_name"]) # un valor por país/indicador
    .pivot(index="country", columns="indicator_name", values="OBS_VALUE")
    .reset_index()
)
df_wide.columns.name = None
print("Columnas disponibles:", df_wide.columns.tolist())

# --- Verificar que todas las variables requeridas existen ---
ADOPTION_VARS = ["gdp_per_capita", "internet_usage", "tertiary_enrollment"]
DEVELOP_VARS  = ["rd_expenditure", "researchers", "hightech_exports"]
ALL_VARS = ADOPTION_VARS + DEVELOP_VARS

# --- Normalización min-max ---
def minmax(s):
    rng = s.max() - s.min()
    if rng == 0:
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / rng

# --- Construcción de índices ---
df_idx = df_wide[["country"]].copy()

df_idx["adoption_index"] = (
    sum(minmax(df_wide[c].fillna(df_wide[c].median())) for c in ADOPTION_VARS)
    / len(ADOPTION_VARS)
)

df_idx["develop_index"] = (
    sum(minmax(df_wide[c].fillna(df_wide[c].median())) for c in DEVELOP_VARS)
    / len(DEVELOP_VARS)
)

df_idx["ai_readiness_index"] = (
    df_idx["adoption_index"] + df_idx["develop_index"]
) / 2

df_idx = df_idx.sort_values("ai_readiness_index", ascending=False).reset_index(drop=True)

display(df_idx.round(3))

Columnas disponibles: ['country', 'gdp_per_capita', 'hightech_exports', 'internet_usage', 'rd_expenditure', 'researchers', 'tertiary_enrollment']


,country,adoption_index,develop_index,ai_readiness_index
0,Uruguay,0.757,0.531,0.644
1,Chile,0.867,0.368,0.618
2,Argentina,0.746,0.476,0.611
3,Brazil,0.281,0.685,0.483
4,Mexico,0.179,0.420,0.299
5,Peru,0.173,0.188,0.181
6,Colombia,0.073,0.137,0.105


## 24. Visualización 1: evolución del acceso a internet

La primera visualización muestra la evolución del porcentaje de individuos que utilizan internet en países seleccionados de América Latina entre 2000 y 2023. Esta dimensión se utiliza como una aproximación a la infraestructura digital disponible para la adopción de tecnologías basadas en inteligencia artificial.

In [ ]:
internet_df = (
    df_case_clean[df_case_clean["indicator_name"] == "internet_usage"]
    .copy()
    .sort_values(["country", "TIME_PERIOD"])
)

highlight = alt.selection_point(fields=["country"], bind="legend")

chart_internet = (
    alt.Chart(internet_df)
    .mark_line(point=True, strokeWidth=3)
    .encode(
        x=alt.X(
            "TIME_PERIOD:Q",
            title="Año",
            axis=alt.Axis(format="d")
        ),
        y=alt.Y(
            "OBS_VALUE:Q",
            title="Usuarios de internet (% de la población)"
        ),
        color=alt.condition(
            highlight,
            alt.Color(
                "country:N",
                title="País",
                sort=country_order,
                scale=alt.Scale(
                    domain=country_order,
                    range=["#1f77b4", "#ff7f0e", "#d62728", "#17becf", "#2ca02c", "#bcbd22", "#8c564b"]
                )
            ),
            alt.value("lightgray")
        ),
        opacity=alt.condition(highlight, alt.value(1), alt.value(0.25)),
        tooltip=[
            alt.Tooltip("country:N", title="País"),
            alt.Tooltip("TIME_PERIOD:Q", title="Año", format=".0f"),
            alt.Tooltip("OBS_VALUE:Q", title="% de la población", format=".2f")
        ]
    )
    .add_params(highlight)
    .properties(
        title="Evolución del acceso a internet en países seleccionados de América Latina (2000–2023)",
        width=800,
        height=420
    )
    .interactive()
)

display(chart_internet)

alt.Chart(...)

La visualización muestra un crecimiento sostenido del acceso a internet en todos los países seleccionados, aunque con ritmos diferentes. La posibilidad de resaltar cada país individualmente permite observar con mayor claridad trayectorias diferenciadas dentro de la región. En conjunto, el gráfico sugiere que la infraestructura digital, una de las condiciones básicas para aprovechar tecnologías basadas en inteligencia artificial, avanzó de forma generalizada, pero no homogénea.

## 25. Visualización 2: relación entre ingreso y conectividad

La segunda visualización compara el PIB per cápita y el acceso a internet en el último año disponible para cada país. El objetivo es explorar si existe una asociación entre capacidad económica y conectividad digital en los países seleccionados.

In [ ]:
latest_internet_scatter = (
    df_case_clean[df_case_clean["indicator_name"] == "internet_usage"]
    .sort_values("TIME_PERIOD")
    .groupby("country", as_index=False)
    .tail(1)[["country", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={
        "TIME_PERIOD": "internet_year",
        "OBS_VALUE": "internet_users"
    })
)

latest_gdp_scatter = (
    df_case_clean[df_case_clean["indicator_name"] == "gdp_per_capita"]
    .sort_values("TIME_PERIOD")
    .groupby("country", as_index=False)
    .tail(1)[["country", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={
        "TIME_PERIOD": "gdp_year",
        "OBS_VALUE": "gdp_per_capita"
    })
)

df_scatter = (
    latest_internet_scatter
    .merge(latest_gdp_scatter, on="country", how="inner")
    .sort_values("gdp_per_capita")
    .reset_index(drop=True)
)

display(df_scatter)

,country,internet_year,internet_users,gdp_year,gdp_per_capita
0,Peru,2023,79.4848,2023,6568.000278
1,Colombia,2023,77.3369,2023,6828.597538
2,Brazil,2023,84.1506,2023,9288.025946
3,Mexico,2023,81.1832,2023,10259.176662
4,Argentina,2023,89.2290,2023,12993.092887
5,Chile,2023,94.4574,2023,14280.347564
6,Uruguay,2023,89.8960,2023,18378.983356


In [ ]:
highlight_scatter = alt.selection_point(fields=["country"], bind="legend", empty="all")
hover_scatter = alt.selection_point(fields=["country"], on="mouseover", empty="none")

base_scatter = alt.Chart(df_scatter).encode(
    x=alt.X(
        "gdp_per_capita:Q",
        title="PIB per cápita (US$ constantes de 2015)"
    ),
    y=alt.Y(
        "internet_users:Q",
        title="Usuarios de internet (% de la población)"
    ),
    color=alt.condition(
        highlight_scatter,
        alt.Color(
            "country:N",
            title="País",
            sort=country_order,
            scale=alt.Scale(
                domain=country_order,
                range=["#1f77b4", "#ff7f0e", "#d62728", "#17becf", "#2ca02c", "#bcbd22", "#8c564b"]
            )
        ),
        alt.value("lightgray")
    ),
    opacity=alt.condition(highlight_scatter, alt.value(1), alt.value(0.3)),
    tooltip=[
        alt.Tooltip("country:N", title="País"),
        alt.Tooltip("gdp_per_capita:Q", title="PIB per cápita", format=",.0f"),
        alt.Tooltip("internet_users:Q", title="% de internet", format=".2f"),
        alt.Tooltip("gdp_year:Q", title="Año PIB", format=".0f"),
        alt.Tooltip("internet_year:Q", title="Año internet", format=".0f"),
    ]
)

points = base_scatter.mark_circle(size=180).add_params(
    highlight_scatter,
    hover_scatter
)

labels = base_scatter.mark_text(
    align="left",
    dx=8,
    dy=-8,
    fontSize=11
).encode(
    text=alt.condition(hover_scatter, "country:N", alt.value(""))
)

chart_scatter = (
    (points + labels)
    .properties(
        title="PIB per cápita y acceso a internet en el último año disponible",
        width=800,
        height=420
    )
    .interactive()
)

display(chart_scatter)

alt.LayerChart(...)

La visualización permite explorar de manera interactiva la relación entre ingreso per cápita y conectividad digital en el último año disponible para cada país. En este caso, el PIB per cápita se expresa en dólares constantes de 2015, lo que mejora la comparabilidad entre observaciones al corregir el efecto de la inflación. En términos generales, los países con mayores niveles de ingreso per cápita tienden a presentar también mayores porcentajes de uso de internet, lo que refuerza la relevancia de la capacidad económica como dimensión estructural de adopción de tecnologías basadas en inteligencia artificial.

## 26. Visualización 3: posicionamiento comparativo por índice de preparación para la IA

La tercera visualización construye un índice exploratorio de preparación estructural
para la IA, combinando dos dimensiones diferenciadas:

- **Adopción** — qué tan listo está el país para incorporar y usar IA:
  capacidad económica (`gdp_per_capita`), conectividad digital (`internet_usage`)
  y formación terciaria (`tertiary_enrollment`).
- **Desarrollo** — qué tan listo está el país para producir e innovar en IA:
  inversión en I+D (`rd_expenditure`), masa crítica investigadora (`researchers`)
  e integración tecnológica (`hightech_exports`).

Para cada dimensión se toma el valor más reciente disponible por país e indicador,
se normalizan las variables a una escala 0–1 mediante min-max y se promedian dentro
de cada sub-índice. El índice compuesto final es el promedio de ambos sub-índices.

Este gráfico no pretende medir de forma definitiva la preparación para la IA, sino
ofrecer una aproximación exploratoria, transparente y reproducible a partir de los
datos disponibles en la API Data360.

In [ ]:
# --- Formato largo para Altair ---
df_plot = df_idx.melt(
    id_vars="country",
    value_vars=["adoption_index", "develop_index", "ai_readiness_index"],
    var_name="index_type",
    value_name="score"
)

# Etiquetas legibles para la leyenda
label_map = {
    "adoption_index":     "Adopción",
    "develop_index":      "Desarrollo",
    "ai_readiness_index": "Índice compuesto",
}
df_plot["Índice"] = df_plot["index_type"].map(label_map)

# Orden de países por índice compuesto descendente
country_order_idx = (
    df_idx.sort_values("ai_readiness_index", ascending=False)["country"].tolist()
)

# --- Línea que conecta adopción y desarrollo por país (dumbbell) ---
line = (
    alt.Chart(df_plot[df_plot["index_type"].isin(["adoption_index", "develop_index"])])
    .mark_line(strokeWidth=2, color="#cccccc")
    .encode(
        y=alt.Y("country:N", sort=country_order_idx, title="País"),
        x=alt.X("score:Q", title="Puntaje (0–1)", scale=alt.Scale(domain=[0, 1])),
        detail="country:N"
    )
)

# --- Puntos ---
color_scale = alt.Scale(
    domain=["Adopción", "Desarrollo", "Índice compuesto"],
    range=["#1f77b4", "#2ca02c", "#d62728"]
)

size_scale = alt.Scale(
    domain=["Adopción", "Desarrollo", "Índice compuesto"],
    range=[120, 120, 200]   # el índice compuesto se destaca con punto más grande
)

points = (
    alt.Chart(df_plot)
    .mark_point(filled=True, opacity=1)
    .encode(
        y=alt.Y("country:N", sort=country_order_idx, title="País"),
        x=alt.X("score:Q", scale=alt.Scale(domain=[0, 1])),
        color=alt.Color(
            "Índice:N",
            scale=color_scale,
            title="Índice",
            legend=alt.Legend(orient="bottom", columns=3)
        ),
        size=alt.Size("Índice:N", scale=size_scale, legend=None),
        tooltip=[
            alt.Tooltip("country:N", title="País"),
            alt.Tooltip("Índice:N",  title="Índice"),
            alt.Tooltip("score:Q",   title="Puntaje", format=".3f"),
        ]
    )
)

# --- Etiquetas del índice compuesto al final de cada fila ---
labels = (
    alt.Chart(df_plot[df_plot["index_type"] == "ai_readiness_index"])
    .mark_text(
        align="center",
        dy=-12,             # sube la etiqueta
        fontSize=11,
        color="#d62728",
        fontWeight="bold"
    )
    .encode(
        y=alt.Y("country:N", sort=country_order_idx),
        x=alt.X("score:Q"),
        text=alt.Text("score:Q", format=".3f"),
    )
)

chart_idx = (
    (line + points + labels)
    .properties(
      title=alt.TitleParams(
            text="Posicionamiento comparativo por índice de preparación para la IA",
            subtitle="Países de América Latina · Comparación entre subíndices de adopción y desarrollo, e índice compuesto como síntesis de ambas dimensiones",
            anchor="start",
            fontSize=14,
            subtitleFontSize=11,
            subtitleColor="#666666"
        ),
        width=620,
        height=320
    )
    .configure_axis(labelFontSize=11, titleFontSize=11)
    .configure_view(stroke=None)
)

display(chart_idx)

alt.LayerChart(...)

**Nota:** el índice compuesto sintetiza conjuntamente las dimensiones de adopción y desarrollo. Por ello, un país puede mostrar un valor relativamente alto en una de las dimensiones y, aun así, ubicarse por debajo de otro en el índice total si su desempeño en la otra dimensión es menor.

## 27. Conclusiones finales

A partir de la exploración de la API Data360 del Banco Mundial, fue posible identificar su estructura general, reconocer la diversidad de bases de datos disponibles y analizar la forma en que la información se organiza en dimensiones temporales, geográficas y temáticas. La exploración inicial permitió comprender el funcionamiento de los endpoints principales y sentó la base metodológica para el desarrollo del caso de estudio.

En la segunda parte del trabajo se propuso un análisis exploratorio sobre la preparación estructural de países de América Latina para aprovechar tecnologías basadas en inteligencia artificial. Dado que esta noción no puede observarse directamente en un único indicador, se construyó una aproximación a partir de **dos dimensiones**:
- **Adopción**: capacidad económica (`gdp_per_capita`), conectividad (`internet_usage`) y formación terciaria (`tertiary_enrollment`).
- **Desarrollo**: inversión en I+D (`rd_expenditure`), masa crítica investigadora (`researchers`) e integración tecnológica (`hightech_exports`).

Las visualizaciones desarrolladas muestran, en primer lugar, una expansión sostenida del acceso a internet en todos los países analizados entre 2000 y 2023, aunque con trayectorias desiguales. En segundo lugar, la comparación entre ingreso per cápita y conectividad digital sugiere una asociación positiva entre ambas variables. Finalmente, el índice exploratorio construido a partir de indicadores normalizados permitió resumir comparativamente la posición relativa de cada país dentro del grupo seleccionado.

Los resultados sugieren que Uruguay y Chile presentan las condiciones estructurales más favorables dentro del conjunto analizado, mientras que Colombia aparece relativamente más rezagada según las variables consideradas. Sin embargo, este resultado debe interpretarse con cautela: se trata de un ejercicio exploratorio y relativo al conjunto de países estudiados, condicionado tanto por la selección de indicadores como por la disponibilidad temporal de los datos.

En síntesis, el trabajo permitió no solo explorar técnicamente la API Data360, sino también utilizarla como fuente para construir un análisis reproducible, visualmente interpretable y metodológicamente argumentado. Como línea futura, este enfoque podría ampliarse incorporando más países, nuevas dimensiones asociadas al desarrollo digital y del capital humano, y metodologías más sofisticadas de construcción de índices compuestos. Asimismo, la lógica desarrollada en este TP ofrece una base sólida para evolucionar hacia una propuesta más ambiciosa orientada a evaluación comparativa y comunicación pública de preparación para la IA.